In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
url="iris_dataset.csv"
name=['sepal-length','sepal-width','petal-length','petal-width','Class']
dataset=pd.read_csv(url,names=names)
dataset.head()
X=dataset.idoc[:,:-1].values
y=dataset.idoc[:,4].values
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20)
from sklearn.neighbors import KNeighborsClassifier
classifier=KNeighborsClassifier(n_neighbors=5)
classifier.fit(X_train,y_train)
y_pred =classifier.predict(X_test)
from sklearn.metrics import classification_report,confusion

class IrisDataset:
    def __init__(self, filepath):
        self.filepath = filepath
        self.data = None
        self.features = None
        self.labels = None
        
    def load_data(self):
        """读取CSV文件"""
        self.data = pd.read_csv(self.filepath)
        # 假设最后一列是标签，其他是特征
        self.features = self.data.iloc[:, :-1].values
        self.labels = self.data.iloc[:, -1].values
        return self.features, self.labels

# 2. KNN分类器
class KNNClassifier:
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None
    
    def fit(self, X_train, y_train):
        """训练模型：只需存储训练数据"""
        self.X_train = X_train
        self.y_train = y_train
    
    def predict(self, X_test):
        """预测新数据"""
        predictions = [self._predict(x) for x in X_test]
        return np.array(predictions)
    
    def _predict(self, x):
        """预测单个样本"""
        # 1. 计算距离
        distances = [self._euclidean_distance(x, x_train) for x_train in self.X_train]
        
        # 2. 获取k个最近邻的索引
        k_indices = np.argsort(distances)[:self.k]
        
        # 3. 获取k个最近邻的标签
        k_nearest_labels = [self.y_train[i] for i in k_indices]
        
        # 4. 投票决定（返回最多的标签）
        most_common = Counter(k_nearest_labels).most_common(1)
        return most_common[0][0]
    
    def _euclidean_distance(self, x1, x2):
        """计算欧氏距离"""
        return np.sqrt(np.sum((x1 - x2) ** 2))

# 3. 主程序
if __name__ == "__main__":
    # 创建数据集对象
    dataset = IrisDataset("iris.csv")  # 替换为你的文件路径
    
    # 加载数据
    X, y = dataset.load_data()
    
    # 划分训练集和测试集
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # 创建KNN分类器
    knn = KNNClassifier(k=3)
    
    # 训练模型
    knn.fit(X_train, y_train)
    
    # 预测
    y_pred = knn.predict(X_test)
    
    # 评估
    accuracy = accuracy_score(y_test, y_pred)
    print(f"准确率: {accuracy:.2%}")
    
    # 测试单个样本
    test_sample = X_test[0].reshape(1, -1)
    prediction = knn.predict(test_sample)
    print(f"测试样本预测结果: {prediction[0]}")
    print(f"测试样本真实标签: {y_test[0]}")

FileNotFoundError: [Errno 2] No such file or directory: 'iris.csv'

In [2]:
import csv
import math
import random

# 1. 数据集类
class IrisDataset:
    def __init__(self, filename="iris_dataset.csv"):
        self.filename = filename
        self.features = []
        self.labels = []
        self.label_map = {}
    
    def load_data(self):
        """从CSV文件加载数据"""
        with open(self.filename, 'r') as file:
            reader = csv.reader(file)
            for row in reader:
                if row:  # 跳过空行
                    # 前4列是特征，最后一列是标签
                    features = [float(x) for x in row[:-1]]
                    label = row[-1]
                    
                    if label not in self.label_map:
                        self.label_map[label] = len(self.label_map)
                    
                    self.features.append(features)
                    self.labels.append(self.label_map[label])
        
        return self.features, self.labels, self.label_map
    
    def split_data(self, test_size=0.2):
        """划分训练集和测试集"""
        data = list(zip(self.features, self.labels))
        random.shuffle(data)
        
        split_idx = int(len(data) * (1 - test_size))
        train_data = data[:split_idx]
        test_data = data[split_idx:]
        
        X_train, y_train = zip(*train_data)
        X_test, y_test = zip(*test_data)
        
        return list(X_train), list(y_train), list(X_test), list(y_test)

# 2. KNN分类器
class KNNClassifier:
    def __init__(self, k=3):
        self.k = k
        self.X_train = []
        self.y_train = []
    
    def fit(self, X_train, y_train):
        """训练模型"""
        self.X_train = X_train
        self.y_train = y_train
    
    def predict(self, X_test):
        """预测"""
        return [self._predict_one(x) for x in X_test]
    
    def _predict_one(self, x):
        """预测单个样本"""
        # 计算距离
        distances = []
        for i, train_x in enumerate(self.X_train):
            dist = 0
            for j in range(len(x)):
                dist += (x[j] - train_x[j]) ** 2
            distances.append((math.sqrt(dist), i))
        
        # 取最近的k个
        distances.sort(key=lambda d: d[0])
        k_nearest = distances[:self.k]
        
        # 统计标签
        label_counts = {}
        for _, idx in k_nearest:
            label = self.y_train[idx]
            label_counts[label] = label_counts.get(label, 0) + 1
        
        # 返回最多的标签
        return max(label_counts.items(), key=lambda x: x[1])[0]

# 3. 主程序
if __name__ == "__main__":
    # 加载数据
    dataset = IrisDataset("iris_dataset.csv")
    features, labels, label_map = dataset.load_data()
    
    print("数据加载完成!")
    print(f"样本数量: {len(features)}")
    print(f"特征维度: {len(features[0])}")
    print(f"标签映射: {label_map}")
    
    # 划分数据集
    X_train, y_train, X_test, y_test = dataset.split_data(test_size=0.2)
    
    print(f"\n训练集: {len(X_train)} 个样本")
    print(f"测试集: {len(X_test)} 个样本")
    
    # 创建并训练KNN模型
    knn = KNNClassifier(k=3)
    knn.fit(X_train, y_train)
    
    # 预测
    y_pred = knn.predict(X_test)
    
    # 计算准确率
    correct = sum(1 for i in range(len(y_test)) if y_test[i] == y_pred[i])
    accuracy = correct / len(y_test)
    
    print(f"\n模型准确率: {accuracy:.2%}")
    
    # 反转标签映射，方便显示
    reverse_map = {v: k for k, v in label_map.items()}
    
    # 显示部分预测结果
    print("\n前5个预测结果:")
    for i in range(min(5, len(y_test))):
        true_name = reverse_map[y_test[i]]
        pred_name = reverse_map[y_pred[i]]
        result = "✓" if y_test[i] == y_pred[i] else "✗"
        print(f"样本{i+1}: 真实={true_name}, 预测={pred_name} {result}")

ValueError: could not convert string to float: 'sepal length (cm)'

In [3]:
import csv
import math
import random

# 1. 数据集类
class IrisDataset:
    def __init__(self, filename="iris_dataset.csv"):
        self.filename = filename
        self.features = []
        self.labels = []
        self.label_map = {}
    
    def load_data(self, has_header=True):
        """从CSV文件加载数据"""
        with open(self.filename, 'r') as file:
            reader = csv.reader(file)
            
            # 跳过表头（如果存在）
            if has_header:
                header = next(reader)
                print(f"表头: {header}")
            
            for row in reader:
                if row:  # 跳过空行
                    # 前4列是特征，最后一列是标签
                    features = [float(x) for x in row[:-1]]
                    label = row[-1]
                    
                    if label not in self.label_map:
                        self.label_map[label] = len(self.label_map)
                    
                    self.features.append(features)
                    self.labels.append(self.label_map[label])
        
        return self.features, self.labels, self.label_map
    
    def split_data(self, test_size=0.2):
        """划分训练集和测试集"""
        data = list(zip(self.features, self.labels))
        random.shuffle(data)
        
        split_idx = int(len(data) * (1 - test_size))
        train_data = data[:split_idx]
        test_data = data[split_idx:]
        
        X_train, y_train = zip(*train_data)
        X_test, y_test = zip(*test_data)
        
        return list(X_train), list(y_train), list(X_test), list(y_test)

# 2. KNN分类器
class KNNClassifier:
    def __init__(self, k=3):
        self.k = k
        self.X_train = []
        self.y_train = []
    
    def fit(self, X_train, y_train):
        """训练模型"""
        self.X_train = X_train
        self.y_train = y_train
    
    def predict(self, X_test):
        """预测"""
        return [self._predict_one(x) for x in X_test]
    
    def _predict_one(self, x):
        """预测单个样本"""
        # 计算距离
        distances = []
        for i, train_x in enumerate(self.X_train):
            dist = 0
            for j in range(len(x)):
                dist += (x[j] - train_x[j]) ** 2
            distances.append((math.sqrt(dist), i))
        
        # 取最近的k个
        distances.sort(key=lambda d: d[0])
        k_nearest = distances[:self.k]
        
        # 统计标签
        label_counts = {}
        for _, idx in k_nearest:
            label = self.y_train[idx]
            label_counts[label] = label_counts.get(label, 0) + 1
        
        # 返回最多的标签
        return max(label_counts.items(), key=lambda x: x[1])[0]

# 3. 主程序
if __name__ == "__main__":
    # 加载数据（跳过表头）
    dataset = IrisDataset("iris_dataset.csv")
    features, labels, label_map = dataset.load_data(has_header=True)
    
    print("数据加载完成!")
    print(f"样本数量: {len(features)}")
    print(f"特征维度: {len(features[0])}")
    print(f"标签映射: {label_map}")
    
    # 划分数据集
    X_train, y_train, X_test, y_test = dataset.split_data(test_size=0.2)
    
    print(f"\n训练集: {len(X_train)} 个样本")
    print(f"测试集: {len(X_test)} 个样本")
    
    # 创建并训练KNN模型
    knn = KNNClassifier(k=3)
    knn.fit(X_train, y_train)
    
    # 预测
    y_pred = knn.predict(X_test)
    
    # 计算准确率
    correct = sum(1 for i in range(len(y_test)) if y_test[i] == y_pred[i])
    accuracy = correct / len(y_test)
    
    print(f"\n模型准确率: {accuracy:.2%}")
    
    # 反转标签映射，方便显示
    reverse_map = {v: k for k, v in label_map.items()}
    
    # 显示部分预测结果
    print("\n前5个预测结果:")
    for i in range(min(5, len(y_test))):
        true_name = reverse_map[y_test[i]]
        pred_name = reverse_map[y_pred[i]]
        result = "✓" if y_test[i] == y_pred[i] else "✗"
        print(f"样本{i+1}: 真实={true_name}, 预测={pred_name} {result}")

表头: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)', 'target_name']
数据加载完成!
样本数量: 150
特征维度: 4
标签映射: {'setosa': 0, 'versicolor': 1, 'virginica': 2}

训练集: 120 个样本
测试集: 30 个样本

模型准确率: 96.67%

前5个预测结果:
样本1: 真实=setosa, 预测=setosa ✓
样本2: 真实=setosa, 预测=setosa ✓
样本3: 真实=virginica, 预测=virginica ✓
样本4: 真实=virginica, 预测=virginica ✓
样本5: 真实=virginica, 预测=virginica ✓


In [4]:
A=np.array([[1,1],[0,1]])
B=np.array([[2,0],[3,4]])
print(A*B)
print(A@B)


[[2 0]
 [0 4]]
[[5 4]
 [3 4]]


In [5]:
import csv
import math
import random

# 1. 数据集类
class IrisDataset:
    def __init__(self, filename="iris_dataset.csv"):
        self.filename = filename
        self.features = []
        self.labels = []
        self.label_map = {}
    
    def load_data(self, has_header=True):
        """从CSV文件加载数据"""
        with open(self.filename, 'r') as file:
            reader = csv.reader(file)
            
            if has_header:
                header = next(reader)
                print(f"表头: {header}")
            
            for row in reader:
                if row:
                    features = [float(x) for x in row[:-1]]
                    label = row[-1]
                    
                    if label not in self.label_map:
                        self.label_map[label] = len(self.label_map)
                    
                    self.features.append(features)
                    self.labels.append(self.label_map[label])
        
        return self.features, self.labels, self.label_map
    
    def split_data(self, test_size=0.2):
        """划分训练集和测试集"""
        data = list(zip(self.features, self.labels))
        random.shuffle(data)
        
        split_idx = int(len(data) * (1 - test_size))
        train_data = data[:split_idx]
        test_data = data[split_idx:]
        
        X_train, y_train = zip(*train_data)
        X_test, y_test = zip(*test_data)
        
        return list(X_train), list(y_train), list(X_test), list(y_test)

# 2. KNN分类器
class KNNClassifier:
    def __init__(self, k=3):
        self.k = k
        self.X_train = []
        self.y_train = []
    
    def fit(self, X_train, y_train):
        """训练模型"""
        self.X_train = X_train
        self.y_train = y_train
    
    def predict(self, X_test):
        """预测"""
        return [self._predict_one(x) for x in X_test]
    
    def _predict_one(self, x):
        """预测单个样本"""
        distances = []
        for i, train_x in enumerate(self.X_train):
            dist = 0
            for j in range(len(x)):
                dist += (x[j] - train_x[j]) ** 2
            distances.append((math.sqrt(dist), i))
        
        distances.sort(key=lambda d: d[0])
        k_nearest = distances[:self.k]
        
        label_counts = {}
        for _, idx in k_nearest:
            label = self.y_train[idx]
            label_counts[label] = label_counts.get(label, 0) + 1
        
        return max(label_counts.items(), key=lambda x: x[1])[0]

# 3. 绘图类
class ResultVisualizer:
    @staticmethod
    def plot_results(y_test, y_pred, label_map):
        """绘制预测结果对比图"""
        # 创建简单的文本绘图
        reverse_map = {v: k for k, v in label_map.items()}
        
        print("\n" + "="*50)
        print("预测结果可视化")
        print("="*50)
        
        # 1. 准确率条状图
        correct = sum(1 for i in range(len(y_test)) if y_test[i] == y_pred[i])
        accuracy = correct / len(y_test)
        
        bar_length = 30
        correct_bars = int(accuracy * bar_length)
        incorrect_bars = bar_length - correct_bars
        
        print(f"\n准确率: {accuracy:.2%}")
        print(f"[{'█' * correct_bars}{'░' * incorrect_bars}]")
        print(f"正确: {correct}/{len(y_test)}  错误: {len(y_test)-correct}/{len(y_test)}")
        
        # 2. 混淆矩阵（简化版）
        print("\n混淆矩阵（前3个类别）:")
        all_labels = sorted(set(y_test) | set(y_pred))
        matrix = {}
        
        for true in all_labels:
            matrix[true] = {}
            for pred in all_labels:
                matrix[true][pred] = 0
        
        for true, pred in zip(y_test, y_pred):
            matrix[true][pred] += 1
        
        # 打印表头
        print("真实\\预测", end="")
        for label in all_labels[:3]:  # 只显示前3个类别
            print(f"  {reverse_map[label]:8}", end="")
        print()
        
        # 打印矩阵
        for true_label in all_labels[:3]:
            print(f"{reverse_map[true_label]:10}", end="")
            for pred_label in all_labels[:3]:
                count = matrix[true_label][pred_label]
                print(f"  {count:8}", end="")
            print()
        
        # 3. 样本预测详情
        print(f"\n详细预测结果（前10个样本）:")
        print("-"*40)
        print("序号 | 真实类别     | 预测类别     | 结果")
        print("-"*40)
        
        for i in range(min(10, len(y_test))):
            true_name = reverse_map[y_test[i]]
            pred_name = reverse_map[y_pred[i]]
            result = "✓" if y_test[i] == y_pred[i] else "✗"
            print(f"{i+1:4} | {true_name:12} | {pred_name:12} | {result}")
        
        # 4. 类别统计
        print(f"\n各类别统计:")
        for label_idx, label_name in reverse_map.items():
            total = sum(1 for y in y_test if y == label_idx)
            correct_count = sum(1 for i in range(len(y_test)) 
                               if y_test[i] == label_idx and y_pred[i] == label_idx)
            if total > 0:
                class_acc = correct_count / total
                print(f"{label_name:12}: {correct_count:2}/{total} ({class_acc:.1%})")
        
        # 5. 简单的直方图
        print(f"\n预测分布:")
        for label_idx, label_name in reverse_map.items():
            count = sum(1 for pred in y_pred if pred == label_idx)
            print(f"{label_name:12}: {'■' * count} ({count})")

# 4. 主程序
if __name__ == "__main__":
    # 加载数据
    dataset = IrisDataset("iris_dataset.csv")
    features, labels, label_map = dataset.load_data(has_header=True)
    
    print("数据加载完成!")
    print(f"样本数量: {len(features)}")
    print(f"特征维度: {len(features[0])}")
    print(f"标签类别: {label_map}")
    
    # 划分数据集
    X_train, y_train, X_test, y_test = dataset.split_data(test_size=0.2)
    
    print(f"\n训练集: {len(X_train)} 个样本")
    print(f"测试集: {len(X_test)} 个样本")
    
    # 创建并训练KNN模型
    knn = KNNClassifier(k=3)
    knn.fit(X_train, y_train)
    
    # 预测
    y_pred = knn.predict(X_test)
    
    # 计算准确率
    correct = sum(1 for i in range(len(y_test)) if y_test[i] == y_pred[i])
    accuracy = correct / len(y_test)
    
    print(f"\n模型准确率: {accuracy:.2%}")
    
    # 绘制结果
    visualizer = ResultVisualizer()
    visualizer.plot_results(y_test, y_pred, label_map)

表头: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)', 'target_name']
数据加载完成!
样本数量: 150
特征维度: 4
标签类别: {'setosa': 0, 'versicolor': 1, 'virginica': 2}

训练集: 120 个样本
测试集: 30 个样本

模型准确率: 96.67%

预测结果可视化

准确率: 96.67%
[█████████████████████████████░]
正确: 29/30  错误: 1/30

混淆矩阵（前3个类别）:
真实\预测  setosa    versicolor  virginica
setosa             4         0         0
versicolor         0         9         1
virginica          0         0        16

详细预测结果（前10个样本）:
----------------------------------------
序号 | 真实类别     | 预测类别     | 结果
----------------------------------------
   1 | virginica    | virginica    | ✓
   2 | versicolor   | versicolor   | ✓
   3 | versicolor   | versicolor   | ✓
   4 | versicolor   | versicolor   | ✓
   5 | virginica    | virginica    | ✓
   6 | versicolor   | versicolor   | ✓
   7 | virginica    | virginica    | ✓
   8 | virginica    | virginica    | ✓
   9 | virginica    | virginica    | ✓
  10 | setosa       | setosa       | ✓

各类别统计:


In [ ]:
A=np.array([[1,1],[0,1]])
B=np.array([[2,0],[3,4]])
print(A*B)
print(A@B)

In [6]:
import numpy as np

# 创建3x5矩阵A
A = np.array([
    [1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10],
    [11, 12, 13, 14, 15]
])

# 创建5x2矩阵B
B = np.array([
    [1, 2],
    [3, 4],
    [5, 6],
    [7, 8],
    [9, 10]
])
print(A*B)
print(A@B)

ValueError: operands could not be broadcast together with shapes (3,5) (5,2) 

In [7]:
import numpy as np

# 创建3x5矩阵A
A = np.array([
    [1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10],
    [11, 12, 13, 14, 15]
])

# 创建5x2矩阵B
B = np.array([
    [1, 2],
    [3, 4],
    [5, 6],
    [7, 8],
    [9, 10]
])

# 只做矩阵乘法运算
result = A @ B
print(result)

[[ 95 110]
 [220 260]
 [345 410]]


In [8]:
class Matrix:
    def __init__(self, d):
        self.d = d
    def mul(self, o):
        r = []
        for i in range(len(self.d)):
            row = []
            for j in range(len(o.d[0])):
                s = 0
                for k in range(len(self.d[0])):
                    s += self.d[i][k] * o.d[k][j]
                row.append(s)
            r.append(row)
        return Matrix(r)

A = Matrix([[1,2,3,4,5],[6,7,8,9,10],[11,12,13,14,15]])
B = Matrix([[1,2],[3,4],[5,6],[7,8],[9,10]])
C = A.mul(B)
print(C.d)

[[95, 110], [220, 260], [345, 410]]
